# Partitioning Strategies: Hash, Range, and Data Lake Partition Layouts

**Module**: 05_partitions_and_shuffle  
**Notebook**: 05_partitioning_strategies.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Hash partitioning vs Range partitioning, bucketing with bucketBy(), and partitioned directory layouts for data lakes.
- **Why does this topic exist?**: Proper storage partitioning enables partition pruning, eliminating 95% of data scans.
- **What problem does it solve?**: High-cardinality directory partitioning creating millions of tiny directories.
- **Where is it used in real-world Data Engineering?**: Designing enterprise Hive / Delta Lake partition architectures.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Shuffle operations, file formats, data lakes |
| **Required depth** | Working knowledge: can choose a partitioning strategy for storage and computation |
| **Data Engineer** | Design table layouts and join keys that reduce scanned data and shuffle volume. |
| **Senior Data Engineer** | Balance partition count, file size, and cardinality for mixed workloads. |
| **Spark specialist** | Reason about hash/range partitioning, bucketing, and partition pruning. |
| **Next topic** | `05_partitions_and_shuffle/06_data_skew_and_solutions.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

There are two different partitioning decisions: how Spark distributes records during a shuffle, and how a table
is laid out on disk. Confusing them leads to directories with millions of tiny files or joins that still shuffle
despite a carefully chosen table layout. A good strategy connects the write layout to the read pattern.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Hash partitioning** | Distribute by key hash | Used by joins and aggregations; even only if keys are even |
| **Range partitioning** | Distribute by sorted ranges | Useful for ordered scans and range predicates |
| **Bucketing** | Pre-shuffle files by hash | Can eliminate a shuffle when both sides use compatible bucketing |
| **Partition pruning** | Skip directories/files | Driven by partition columns in the filter |
| **Cardinality** | Number of distinct key values | High cardinality can create too many partitions |
| **Over-partitioning** | Too many small partitions | Metadata and task overhead dominate |

### How it works, step by step

1. The writer chooses partition columns or a bucketing specification.
2. Each output directory or bucket contains rows for one partition value or hash bucket.
3. A reader with a filter on a partition column prunes directories before reading files.
4. A join can avoid a shuffle only when both sides are bucketed compatibly and the join key matches.
5. Hash partitioning during a shuffle uses `Murmur3Hash` by default for SQL expressions.
6. AQE may coalesce partitions after a shuffle, but it does not change the on-disk layout.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Partitioning Strategies

1. **Hash Partitioning**: `hash(key) % N` (Used for joins/aggregations)
2. **Range Partitioning**: Quantile sample ranges (Used for `orderBy`)
3. **Directory Partitioning**: `year=2026/month=01/` (Used in data lakes)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Partitioning_Strategies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
# Write partitioned by channel
out_path = "/opt/spark-data/warehouse/partitioned_by_channel"
orders.write.mode("overwrite").partitionBy("channel").parquet(out_path)

print("Partitions created on disk:")
for p in os.listdir(out_path):
    if not p.startswith("."):
        print(f"  Directory: {p}")

---

## Experiment: Partition Pruning Verification

- **1. What are we testing?**: Querying with filter on partition column skips other folders entirely.
- **2. What do we expect?**: Physical plan shows PartitionFilters: [channel ='Mobile'].

In [ ]:
read_part = spark.read.parquet(out_path)
filtered = read_part.filter("channel = 'Mobile'")
filtered.explain("formatted")

**3. What happened?**: PartitionFilters pruned irrelevant channel directories at the file scan level.  
**4. Why did it happen?**: Spark uses directory metadata to skip non-matching partition folders.  
**5. Practical Engineering Lesson**: Partition by low-to-medium cardinality columns used frequently in WHERE clauses.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Designing a clickstream lake layout

- **Industry**: Advertising technology
- **Scenario**: Billions of ad events arrive hourly and analysts filter by date, campaign, and country.
- **Problem**: A single directory of files forces every query to scan the full history.
- **Why this concept is relevant**: Partitioning by date and campaign enables pruning for the dominant filters; country may be better as a cluster/bucket key if cardinality is high.
- **Implementation idea**: Measure query scan bytes for representative filters under candidate layouts before committing.
- **Expected result**: A layout that scans only the relevant dates and campaigns.
- **Alternative**: Use a table format with hidden partitioning and clustering.
- **Trade-offs**: Native directory partitioning is portable; a table format adds pruning and evolution features at the cost of another dependency.

### Use Case 2: Bucketing a recurring join

- **Industry**: Telecom
- **Scenario**: Two large tables are joined nightly on `subscriber_id`.
- **Problem**: Both sides shuffle every night even though the join key and data are stable.
- **Why this concept is relevant**: Bucketing both tables by the same key with a compatible bucket count can remove the shuffle.
- **Implementation idea**: Write bucketed tables, then compare the join plan with and without the shuffle.
- **Expected result**: A plan without the exchange, or a measured decision that bucketing is not worth it.
- **Alternative**: Broadcast the smaller dimension if it fits.
- **Trade-offs**: Bucketing helps repeated large-large joins; broadcasting is simpler when one side is small.

---

## Business Value

- Less data scanned per query lowers compute cost.
- Stable layouts reduce nightly runtime variance.
- Better self-service analytics through predictable partition pruning.
- Clear ownership of layout decisions between ingestion and consumption teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Partition on columns used in frequent, selective filters, not on every column.
- Keep partition cardinality moderate; avoid user IDs or timestamps at second granularity.
- Target file sizes rather than a fixed number of files.
- Verify that the reader actually prunes; inspect `PartitionFilters` in the plan.
- Bucketing only helps when both sides are written with compatible specifications.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare partition layouts by scan bytes

**Steps**

1. Write the same dataset partitioned by a low-cardinality column and by a high-cardinality column.
2. Run a selective filter on each layout.
3. Compare files scanned, bytes read, and task count in the UI.

**What to observe**: The low-cardinality layout prunes cleanly; the high-cardinality layout creates many small files and metadata overhead.

**Healthy result**: The chosen layout matches the dominant query filter.

**Problematic result**: The plan reads all partitions because the filter column is not a partition column.

**Engineering takeaway**: Layout follows access patterns, not aesthetics.

---

## Trade-offs and Alternatives

### Advantages

- Large scan reductions
- Reusable join layout
- Portable across engines

### Limitations

- Layout is hard to change later
- High cardinality creates small files
- Bucketing has strict compatibility rules

### When to use it

- Date-filtered lake queries
- Repeated large joins
- Columnar analytics

### When not to use it

- Partitioning by unique IDs
- Bucketing one side only
- Changing bucket counts casually

### Alternatives inside Spark

- Directory partitioning
- Bucketing
- AQE
- Partition pruning

### Alternatives outside Spark

- Iceberg/Delta/Hudi partitioning
- Warehouse clustering
- Search indexes

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Partition by Year and Month
**Problem Statement**: Write orders partitioned by year and month.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Directory Cardinality
**Problem Statement**: Calculate distinct values of potential partition column.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Bucketing Concept
**Problem Statement**: Explain bucketing vs partitioning.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Bucketing that never eliminates the shuffle

**Scenario**: A team buckets two tables by the same column but the join still shuffles.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the bucket counts and the join key expression.
2. Confirm both tables were written with `bucketBy` and read as tables, not raw paths.
3. Verify that the join uses the exact same key and compatible types.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Partition by Year and Month):
```python
orders.withColumn('yr', F.year('order_ts')).write.mode('overwrite').partitionBy('yr').parquet('/opt/spark-data/warehouse/part_yr')
```

#### Solution 2 (Check Directory Cardinality):
```python
print('Channel distinct count:', orders.select('channel').distinct().count())
```

#### Solution 3 (Bucketing Concept):
```python
print('Partitioning creates directories; bucketing creates fixed-count hash files within directories.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partition Pruning**: Skips entire folders based on WHERE filters.
- **Cardinality Rule**: Never partition by high cardinality keys (e.g. timestamp, id).

### Essential Technical Terms
- `Partition Pruning`: Skipping storage directories at read time.
- `Bucketing`: Pre-shuffling into fixed buckets.

### Common Engineering Mistakes
- **Partitioning by customer_id**: Creates 100,000 tiny directories, destroying metadata performance.

### Interview Questions on This Topic
- **Q: What is partition pruning?**
  - *A*: Spark skipping entire directory partitions during scanning because the query filter eliminates them.


### More Interview Questions

- **Q: When does bucketing eliminate a join shuffle?**
  - *A*: When both sides are bucketed by the join key with a compatible bucket count and the planner can prove the layout matches.
- **Q: Why is partitioning by user ID usually a bad idea?**
  - *A*: It creates a very large number of small partitions/files, increasing metadata and task overhead and often scanning many files.

### What to Study Next
Open 06_data_skew_and_solutions.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')